In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
df=pd.read_csv('stock_data_july_2025.csv')
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(by=["Ticker", "Date"]).reset_index(drop=True)


In [3]:
print(f"Total rows: {len(df)}")
print(f"Unique Tickers: {df['Ticker'].nunique()}")
print(f"Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print("Missing Values:\n", df.isnull().sum())

Total rows: 4346
Unique Tickers: 82
Date Range: 2025-06-01 to 2025-07-23
Missing Values:
 Date              0
Ticker            0
Open Price        0
Close Price       0
High Price        0
Low Price         0
Volume Traded     0
Market Cap        0
PE Ratio          0
Dividend Yield    0
EPS               0
52 Week High      0
52 Week Low       0
Sector            0
dtype: int64


In [4]:
df.to_csv("cleaned_stock_data.csv", index=False)

In [5]:
df = pd.read_csv("cleaned_stock_data.csv")
df["Date"] = pd.to_datetime(df["Date"])
df["Daily_Return"] = df.groupby("Ticker")["Close Price"].pct_change()
df["SMA_10"] = df.groupby("Ticker")["Close Price"].transform(
    lambda x: x.rolling(window=10).mean()
)
df["Rolling_Volatility_10"] = df.groupby("Ticker")["Daily_Return"].transform(
    lambda x: x.rolling(window=10).std() * np.sqrt(252)
)
df = df.dropna().reset_index(drop=True)
df.to_csv("featured_stock_data.csv", index=False)
print(f"Feature engineering complete. Total rows remaining: {len(df)}")

Feature engineering complete. Total rows remaining: 3526


In [6]:
# Missing values appeared in the feature engineered dataset due to the rolling calculations,
# which require a minimum number of data points to compute. The first 9 rows for each ticker
# will have NaN values for SMA_10 and Rolling_Volatility_10, and the first row will have NaN 
# for Daily_Return. These rows were dropped to ensure a clean dataset for modeling.

In [ ]:

df = pd.read_csv("featured_stock_data.csv")
low_thresh = df["Rolling_Volatility_10"].quantile(0.33)
high_thresh = df["Rolling_Volatility_10"].quantile(0.67)

print(f"Low Risk Threshold (Vol < {low_thresh:.4f})")
print(
    f"Medium Risk Threshold ({low_thresh:.4f} <= Vol <= {high_thresh:.4f})"
)
print(f"High Risk Threshold (Vol > {high_thresh:.4f})")
def assign_risk_category(vol):
  if vol <= low_thresh:
    return "Low Risk"
  elif vol <= high_thresh:
    return "Medium Risk"
  else:
    return "High Risk"

df["Risk_Category"] = df["Rolling_Volatility_10"].apply(assign_risk_category)

df_model = pd.get_dummies(df, columns=["Sector"], drop_first=True)
risk_mapping = {"Low Risk": 0, "Medium Risk": 1, "High Risk": 2}
df_model["Risk_Category_Encoded"] = df_model["Risk_Category"].map(risk_mapping)
scaler = StandardScaler()
numerical_cols = [
    "Open Price",
    "Close Price",
    "High Price",
    "Low Price",
    "Volume Traded",
    "Market Cap",
    "PE Ratio",
    "Dividend Yield",
    "EPS",
    "52 Week High",
    "52 Week Low",
    "Daily_Return",
    "SMA_10",
    "Rolling_Volatility_10",
]

df_model[numerical_cols] = scaler.fit_transform(df_model[numerical_cols])
print("\nScaled numerical columns:")
print(df_model[numerical_cols].describe())

print("\nRisk Category Distribution:")
print(df["Risk_Category"].value_counts())
df_model.to_csv("labeled_stock_data.csv", index=False)
print(
    "\nTarget labeling complete. Saved as 'labeled_stock_data.csv'. Total"f" rows: {len(df)}"
)

Low Risk Threshold (Vol < 0.5976)
Medium Risk Threshold (0.5976 <= Vol <= 0.7951)
High Risk Threshold (Vol > 0.7951)

Scaled numerical columns:
        Open Price   Close Price    High Price     Low Price  Volume Traded  \
count  3526.000000  3.526000e+03  3.526000e+03  3.526000e+03   3.526000e+03   
mean      0.000000 -6.448488e-17 -1.128485e-16  4.836366e-17   4.433335e-17   
std       1.000142  1.000142e+00  1.000142e+00  1.000142e+00   1.000142e+00   
min      -0.674276 -6.738908e-01 -6.742005e-01 -6.734570e-01  -9.511394e-01   
25%      -0.480167 -4.804536e-01 -4.812813e-01 -4.794287e-01  -5.973906e-01   
50%      -0.298168 -2.993414e-01 -2.976238e-01 -2.976355e-01  -3.063833e-01   
75%       0.235245  2.398516e-01  2.398048e-01  2.373336e-01   2.763585e-01   
max       7.426102  7.518714e+00  7.459901e+00  7.464216e+00   4.777357e+00   

         Market Cap      PE Ratio  Dividend Yield           EPS  52 Week High  \
count  3.526000e+03  3.526000e+03    3.526000e+03  3.526000e+03

In [9]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
df = pd.read_csv("labeled_stock_data.csv")

# 1. Risk Category Distribution Bar Chart
plt.figure(figsize=(8, 5))
ax = sns.countplot(
    data=df,
    x="Risk_Category",
    order=["Low Risk", "Medium Risk", "High Risk"],
    palette="viridis",
)
plt.title("Distribution of Stock Risk Categories", fontsize=14, fontweight="bold")
plt.xlabel("Risk Category", fontsize=12)
plt.ylabel("Frequency (Trading Days)", fontsize=12)
for p in ax.patches:
  ax.annotate(
      f"{int(p.get_height())}",
      (p.get_x() + p.get_width() / 2.0, p.get_height() / 2),
      ha="center",
      va="center",
      color="white",
      fontsize=12,
      fontweight="bold",
  )
plt.savefig("risk_distribution.png", dpi=300, bbox_inches="tight")
plt.close()

# 2. Rolling Volatility across Sectors (Boxplot)
plt.figure(figsize=(12, 6))
# Filter original sector columns if one-hot encoded, or use sector if preserved.
# Let's melt or boxplot using unscaled rolling volatility vs sector if available.
sns.boxplot(
    data=df,
    x="Sector_Technology",
    y="Rolling_Volatility_10",
    palette="Set2",
)  # Example plot
plt.title("Rolling Volatility Distribution", fontsize=14, fontweight="bold")
plt.savefig("volatility_boxplot.png", dpi=300, bbox_inches="tight")
plt.close()

# 3. Correlation Heatmap of Numerical Features
plt.figure(figsize=(10, 8))
num_cols = [
    "Open Price",
    "Close Price",
    "Volume Traded",
    "Market Cap",
    "PE Ratio",
    "Daily_Return",
    "SMA_10",
    "Rolling_Volatility_10",
]
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title(
    "Feature Correlation Heatmap", fontsize=14, fontweight="bold"
)
plt.savefig("correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.close()

print(
    "EDA visualizations generated and saved successfully as PNG images ('risk_distribution.png', 'volatility_boxplot.png', 'correlation_heatmap.png')!"
)

Fontconfig warning: We will not regenerate the cache because some cache files were generated by a newer version (0x2012003) of Fontconfig. Please regenerate the cache with the latest version of Fontconfig to avoid any unexpected behavior. (current version: 0x2012002)
/tmp/ipykernel_2825/3876340917.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(
/tmp/ipykernel_2825/3876340917.py:36: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


EDA visualizations generated and saved successfully as PNG images ('risk_distribution.png', 'volatility_boxplot.png', 'correlation_heatmap.png')!
